In [28]:
import pandas as pd
from java_post_validation import check_existence
from pymongo import MongoClient
from bson import ObjectId

client = MongoClient("127.0.0.1", 27017)
db = client["bridge"]
col = db["java_api_call_changes"]
df = pd.read_excel("../../benchmark/ground_truth/java_record_samples.xlsx")
dest_folder = "/data/kyle/packages"
pos_df = df[df["index_after"].notna()]
len(pos_df)

346

In [29]:
def get_api_info(oid: str, idx: int, old: bool = True):
    doc = col.find_one({"_id": ObjectId(oid)})
    if old:
        api_call = doc["old_callees"][idx]
    else:
        api_call = doc["new_callees"][idx]
    return api_call["full_name"], api_call["arguments"]


def check_candidates(api_sigs: list[dict]):
    if len(api_sigs) == 0:
        return "Phantom"
    if len(api_sigs) == 1:
        return "One"


def process_one_row(row):
    oid = row["_id"]
    library = row["library"]
    version_before = row["version_before"]
    version_after = row["version_after"]
    index_before = int(row["index_before"])
    index_after = int(row["index_after"])

    parts_before = [int(_) for _ in version_before.split(".")]
    parts_after = [int(_) for _ in version_after.split(".")]
    reverse = parts_before > parts_after

    api_fqn_before, arguments_before = get_api_info(oid, index_before)
    sigs_before = check_existence(
        library, version_before, dest_folder, api_fqn_before, len(arguments_before)
    )
    sigs_before = [sig["parameter_types"] for sig in sigs_before]
    api_fqn_after, arguments_after = get_api_info(oid, index_after, False)
    sigs_after = check_existence(
        library, version_after, dest_folder, api_fqn_after, len(arguments_after)
    )
    sigs_after = [sig["parameter_types"] for sig in sigs_after]

    legacy_api, new_api = row["legacy_api"], row["new_api"]
    if reverse:
        legacy_api, new_api = new_api, legacy_api

    res = [
        {
            "_id": oid,
            "api_fqn": api_fqn_before,
            "arguments": arguments_before,
            "candidates": sigs_before,
            "true": legacy_api.split("(")[1][:-1].split(", "),
        },
        {
            "_id": oid,
            "api_fqn": api_fqn_after,
            "arguments": arguments_after,
            "candidates": sigs_after,
            "true": new_api.split("(")[1][:-1].split(", "),
        },
    ]
    return res


res = []
for row in pos_df.to_dict("records"):
    res.extend(process_one_row(row))

In [30]:
not_found = []
exact_match = []
multi_cands = []
for doc in res:
    if len(doc["candidates"]) == 0:
        not_found.append(doc)
    elif len(doc["candidates"]) == 1:
        exact_match.append(doc)
    else:
        multi_cands.append(doc)

In [31]:
len(not_found), len(exact_match), len(multi_cands)

(40, 578, 74)

In [32]:
import json

with open("test.json", "w") as outf:
    json.dump(multi_cands, outf, indent=2)

In [8]:
from java_api_signature_resolver import best_cand

In [9]:
import json

multi_cands = json.load(open("./test.json"))
len(multi_cands)

74

In [10]:
for c in multi_cands:
    bc = best_cand(c["arguments"], c["candidates"])
    if bc != c["true"]:
        print(c["_id"], bc, c["true"])

695fbdd727ebe30d37aad0df ['String', 'TransactionConfig'] ['String', 'Value']
695fbdd727ebe30d37aad0df ['String', 'TransactionConfig'] ['String', 'Value']
695fbdd727ebe30d37a9ba14 ['int', 'int'] ['int[]', 'int']
695fbdd727ebe30d37a9ba14 ['int', 'int'] ['int[]', 'int']
